# Getting started with HiData

This tutorial builds a small `HiData` from scratch and shows its levels, selections, how information moves
between levels, and how the object is written to disk. The data is simulated.

In [1]:
import numpy as np
import pandas as pd
from anndata import AnnData

import hidata
from hidata import HiData

rng = np.random.default_rng(0)

## Building a HiData

A `HiData` is built from three parts. The group table has one row per group, the subgroup table one row per
subgroup with the group of every subgroup, and the cells are an AnnData with the subgroup of every cell in `obs`. The
subgroup table is optional; without it, `C.obs` names the group of every cell. Here the groups are donors and the
subgroups their samples, so `group_id` and `subgroup_id` name the donor and sample columns.

In [2]:
donors = pd.DataFrame(
    {"age": [34, 52, 61, 70], "disease": ["healthy", "healthy", "covid", "covid"]},
    index=pd.Index(["P1", "P2", "P3", "P4"], name="donor"),
)
samples = pd.DataFrame(
    {"donor": ["P1", "P2", "P3", "P3", "P4", "P4"], "day": [0, 0, 0, 7, 0, 7]},
    index=pd.Index(["P1_d0", "P2_d0", "P3_d0", "P3_d7", "P4_d0", "P4_d7"], name="sample"),
)
n_cells = 600
obs = pd.DataFrame(
    {"sample": rng.choice(samples.index, n_cells), "cell_type": rng.choice(["B", "T", "Monocyte"], n_cells)},
    index=[f"cell_{i}" for i in range(n_cells)],
)
cells = AnnData(
    rng.poisson(1.0, (n_cells, 20)).astype(np.float32),
    obs=obs,
    var=pd.DataFrame(index=[f"gene_{i}" for i in range(20)]),
)
cells.obsm["X_pca"] = rng.normal(size=(n_cells, 5))

hd = HiData(G=donors, S=samples, C=cells, group_id="donor", subgroup_id="sample")
hd

╔═ HiData(n_groups=4, n_subgroups=6, n_cells=600) ═════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃                        ┃ S (subgroups, 1..2 per ┃ C (cells, 88..114 per  ┃ ║
║ ┃ G (groups)             ┃ group)                 ┃ subgroup)              ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ DataFrame with 4       │ DataFrame with 6       │ View of AnnData object │ ║
║ │ groups × 2 columns     │ subgroups × 2 columns  │ with n_obs × n_vars =  │ ║
║ │                        │                        │ 600 × 20               │ ║
║ │     columns: 'age',    │     columns: 'donor',  │     obs: 'sample',     │ ║
║ │ 'disease'              │ 'day'                  │ 'cell_type'            │ ║
║ │                        │                        │     obsm: 'X_pca'      │ ║
║ └────────────────────────┴────────────────────────┴────────────────────────┘ ║
╚═ group_id='donor', subgroup_id='sample' ═════════════════════════════════════╝

## Levels

The levels are available as `G`, `S` and `C`. Tables are returned as DataFrames, AnnData and MuData levels as they
are.

In [3]:
hd.G

,age,disease
donor,,
P1,34,healthy
P2,52,healthy
P3,61,covid
P4,70,covid


In [4]:
hd.S

,donor,day
sample,,
P1_d0,P1,0
P2_d0,P2,0
P3_d0,P3,0
P3_d7,P3,7
P4_d0,P4,0
P4_d7,P4,7


In [5]:
hd.n_groups, hd.n_subgroups, hd.n_cells

(4, 6, 600)

Cells are stored grouped by subgroup, and subgroups grouped by group, in the row order of the level above. The
cells of a unit are a contiguous block of `C`.

In [6]:
hd.cells_of("P3_d7", level="subgroup")

View of AnnData object with n_obs × n_vars = 99 × 20
    obs: 'sample', 'cell_type'
    obsm: 'X_pca'
    layers: None (.X)

## Selecting

A selection at one level selects the matching units at the other levels. Selecting groups keeps their subgroups and
cells, and selecting subgroups or cells drops the units above that are left without members.

A selection can be a condition, a list of names or a mask of `True` and `False`:

- A mask that is a pandas `Series` is matched to the groups, subgroups or cells by name. It can be built from the
  `HiData` or from the tables it was made from, even though their rows are in another order.
- A mask that is a plain array or list is read by position, in the order of the `HiData`.
- A `Series` without names, such as `pd.Series(array)` with the automatic index 0, 1, 2, ..., raises an error,
  because it could be in either order. Pass `mask.values` if it is in the order of the `HiData`.
- A missing value in a mask counts as "not selected".
- `filter_groups`, `filter_subgroups` and `filter_cells` take names as text, for example `["P1", "P3"]`, and raise
  an error for a name that does not exist. They do not take integers or slices: select by position with `hd[...]`
  or `sel`.

In [7]:
def sizes(data):
    return {"donors": data.n_groups, "samples": data.n_subgroups, "cells": data.n_cells}


pd.DataFrame(
    {
        "all": sizes(hd),
        "covid donors": sizes(hd.filter_groups("disease == 'covid'")),
        "day 7 samples": sizes(hd.filter_subgroups("day == 7")),
        "B cells": sizes(hd.filter_cells("cell_type == 'B'")),
    }
)

,all,covid donors,day 7 samples,B cells
donors,4,2,2,4
samples,6,4,2,6
cells,600,412,213,198


## Moving information between levels

`get_df` collects variables from any level into a table at the requested level. Group and subgroup columns are
repeated for their cells, and cell values are averaged for their subgroups or groups. Keys can be columns, genes or
keys of `obsm`.

In [8]:
hd.get_df(["age", "day", "cell_type", "gene_0"], level="cell").head()

,age,day,cell_type,gene_0
cell_5,34,0,B,0.0
cell_6,34,0,B,1.0
cell_7,34,0,Monocyte,2.0
cell_23,34,0,T,2.0
cell_27,34,0,T,0.0


In [9]:
hd.get_df(["disease", "gene_0", "X_pca"], level="subgroup")

,disease,gene_0,X_pca_0,X_pca_1,X_pca_2,X_pca_3,X_pca_4
sample,,,,,,,
P1_d0,healthy,1.050000,-0.068292,-0.049820,0.048927,-0.019036,0.030066
P2_d0,healthy,1.113636,0.146336,-0.136747,-0.074034,-0.000194,0.123579
P3_d0,covid,0.988889,0.062226,-0.109444,-0.131497,-0.053159,-0.137758
P3_d7,covid,1.121212,0.001638,-0.139523,-0.082912,-0.072922,0.155458
P4_d0,covid,1.018349,0.023926,0.128600,0.032593,0.008040,-0.253796
P4_d7,covid,0.929825,0.095759,0.204614,-0.084036,-0.003433,0.101666


Summaries of the cells, such as cell type compositions or pseudobulk profiles, are stored in `obsm` of their level.
They are subset together with the level.

In [10]:
hd.composition("cell_type", level="subgroup")
hd.aggregate(key_added="pseudobulk", level="subgroup")
hd.levels["subgroup"]

AnnData object with n_obs × n_vars = 6 × 0
    obs: 'donor', 'day'
    obsm: 'cell_type_fractions', 'pseudobulk'

In [11]:
covid = hd.filter_groups("disease == 'covid'")
covid.levels["subgroup"].obsm["cell_type_fractions"]

cell_type,B,Monocyte,T
sample,,,
P3_d0,0.277778,0.344444,0.377778
P3_d7,0.303030,0.323232,0.373737
P4_d0,0.339450,0.321101,0.339450
P4_d7,0.342105,0.333333,0.324561


## From and to a single AnnData

`to_anndata` attaches the group and subgroup columns to every cell, and `from_anndata` splits such an AnnData into
levels again, moving the columns that are constant within groups or subgroups into their tables.

In [12]:
flat = hd.to_anndata()
HiData.from_anndata(flat, group_id="donor", subgroup_id="sample")

╔═ HiData(n_groups=4, n_subgroups=6, n_cells=600) ═════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃                        ┃ S (subgroups, 1..2 per ┃ C (cells, 88..114 per  ┃ ║
║ ┃ G (groups)             ┃ group)                 ┃ subgroup)              ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ DataFrame with 4       │ DataFrame with 6       │ AnnData object with    │ ║
║ │ groups × 2 columns     │ subgroups × 2 columns  │ n_obs × n_vars = 600 × │ ║
║ │                        │                        │ 20                     │ ║
║ │     columns: 'age',    │     columns: 'donor',  │     obs: 'sample',     │ ║
║ │ 'disease'              │ 'day'                  │ 'cell_type', 'donor'   │ ║
║ │                        │                        │     obsm: 'X_pca'      │ ║
║ └────────────────────────┴────────────────────────┴────────────────────────┘ ║
╚═ group_id='donor', subgroup_id='sample' ═════════════════════════════════════╝

Cells and subgroups whose group or subgroup is missing are dropped, and a warning names them. A string that
stands for a missing value, such as `"nan"`, `"None"` or `"NA"`, counts as missing too. Below, the first two cells
of `flat` lose their sample, once as `None` and once as the string `"nan"`. If such strings are real names in your
data, rename them before building the `HiData`.

In [13]:
unassigned = flat.copy()
unassigned.obs[["donor", "sample"]] = unassigned.obs[["donor", "sample"]].astype(object)
unassigned.obs.iloc[:2, unassigned.obs.columns.get_indexer(["donor", "sample"])] = [[None, None], ["nan", "nan"]]
HiData.from_anndata(unassigned, group_id="donor", subgroup_id="sample").n_cells, flat.n_obs

Dropping 2 cells with a missing 'sample' (empty, or a value such as NaN, None, 'nan' or 'NA'): 'cell_5' (None), 'cell_6' ('nan'). If these are real names, rename them first.


(598, 600)

`nest` and `collapse` turn the values of a group table column into the new groups, so that column is treated like
an identifier. Below, every donor gets a hospital, and the hospital of `P4` was entered as the string `"NA"`.

In [14]:
by_donor = HiData.from_anndata(flat, group_id="donor")
by_donor.obs["hospital"] = ["Munich", "Munich", "Berlin", "NA"]
by_donor.G

,age,disease,hospital
donor,,,
P1,34,healthy,Munich
P2,52,healthy,Munich
P3,61,covid,Berlin
P4,70,covid,NA


`nest` makes the hospitals the groups and the donors their subgroups. `P4` has no hospital, so it is dropped
with its cells, and the warning names it.

In [15]:
nested = by_donor.nest("hospital")
nested

Dropping 1 groups with a missing 'hospital' (empty, or a value such as NaN, None, 'nan' or 'NA'): 'P4' ('NA'). If these are real names, rename them first.


╔═ HiData(n_groups=2, n_subgroups=3, n_cells=377) ═════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃                        ┃ S (subgroups, 1..2 per ┃ C (cells, 88..189 per  ┃ ║
║ ┃ G (groups)             ┃ group)                 ┃ subgroup)              ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ DataFrame with 2       │ DataFrame with 3       │ View of AnnData object │ ║
║ │ groups × 1 columns     │ subgroups × 2 columns  │ with n_obs × n_vars =  │ ║
║ │                        │                        │ 377 × 20               │ ║
║ │     columns: 'disease' │     columns: 'age',    │     obs: 'sample',     │ ║
║ │                        │ 'hospital'             │ 'cell_type', 'donor',  │ ║
║ │                        │                        │ 'day'                  │ ║
║ │                        │                        │     obsm: 'X_pca'      │ ║
║ └────────────────────────┴────────────────────────┴────────────────────────┘ ║
╚═ group_id='hospital', subgroup_id='donor' ═══════════════════════════════════╝

In [16]:
nested.S

,age,hospital
donor,,
P3,61,Berlin
P1,34,Munich
P2,52,Munich


`collapse` merges the donors of a hospital into one group and drops `P4` in the same way. Rename the value first if
`"NA"` is a real hospital. A missing value in other grouping columns drops nothing: the splits and
`subsample_cells` treat it as a value of its own, and `composition` and `pseudobulk` count those cells as the
cell group `"nan"`.

In [17]:
by_donor.collapse("hospital").G

Dropping 1 groups with a missing 'hospital' (empty, or a value such as NaN, None, 'nan' or 'NA'): 'P4' ('NA'). If these are real names, rename them first.


,disease,n_donor
hospital,,
Berlin,covid,1
Munich,healthy,2


## Writing and reading

A `HiData` is written to an HDF5 file or a Zarr store, see the {doc}`specification <../io/spec>`.

In [18]:
hd.write_hd("data/getting_started.hd.zarr")
hidata.io.read_hd("data/getting_started.hd.zarr")

╔═ HiData(n_groups=4, n_subgroups=6, n_cells=600) ═════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃                        ┃ S (subgroups, 1..2 per ┃ C (cells, 88..114 per  ┃ ║
║ ┃ G (groups)             ┃ group)                 ┃ subgroup)              ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ DataFrame with 4       │ DataFrame with 6       │ AnnData object with    │ ║
║ │ groups × 2 columns     │ subgroups × 2 columns  │ n_obs × n_vars = 600 × │ ║
║ │                        │                        │ 20                     │ ║
║ │     columns: 'age',    │     columns: 'donor',  │     obs: 'sample',     │ ║
║ │ 'disease'              │ 'day'                  │ 'cell_type'            │ ║
║ │                        │     obsm:              │     obsm: 'X_pca'      │ ║
║ │                        │ 'cell_type_fractions', │                        │ ║
║ │                        │ 'pseudobulk'           │                        │ ║
║ └────────────────────────┴────────────────────────┴────────────────────────┘ ║
╚═ group_id='donor', subgroup_id='sample' ═════════════════════════════════════╝